# 03 Follow-ups

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['GE_DRIVE_ROOT'] = '/content/drive/MyDrive/hypothesaes_goemotions'
REPO_URL = 'https://github.com/YOUR_USERNAME/HypotheSAEs.git'   # <-- your fork
REPO_DIR = '/content/HypotheSAEs'

## Install

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

!pip install -q -e {REPO_DIR}
!pip install -q -U datasets sentence-transformers transformers scikit-learn statsmodels "numpy<2.0.0"
!pip install -q -U vllm torchaudio torchvision nvidia-cuda-runtime-cu12
!pip uninstall -y -q torchcodec

# editable-install .pth files are only read at interpreter startup, so add the paths directly
import sys, importlib
for p in (REPO_DIR, os.path.join(REPO_DIR, 'goemotions')):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()

## Imports

In [ ]:
import numpy as np
from scipy.stats import pearsonr, spearmanr
import ge_pipeline as ge
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.annotate import annotate_texts_with_concepts
from hypothesaes.evaluation import score_hypotheses
ge.set_seed()
ge.redirect_annotation_cache()
print('free VRAM (GB):', ge.gpu_free_gb())

## Load Heldout and Hypotheses

In [ ]:
ds, NAMES = ge.load_goemotions()
test_texts = list(ds['test']['text'])
test_targets = ge.build_targets(ds['test']['labels'], NAMES)

interp = ge.read_json('06_interpretations')
evaluation = ge.read_json('07_evaluation')
hidx = evaluation['heldout_indices']            # same rows notebook 02 annotated
held_texts = [test_texts[i] for i in hidx]
held_targets = {t: test_targets[t][hidx] for t in test_targets}

selected = {t: [int(n) for n in v] for t, v in interp['selected'].items()}
neuron2hyp = {int(k): v['interpretation'] for k, v in interp['best'].items() if v['interpretation']}
union_hyps = sorted(set(neuron2hyp.values()))
len(held_texts), len(union_hyps)

## Cosine Annotation

In [ ]:
te = get_local_embeddings(held_texts, model=ge.EMBEDDER, nomic_prefix='search_document: ',
                          cache_name='followup_held_doc', batch_size=128)
he = get_local_embeddings(union_hyps, model=ge.EMBEDDER, nomic_prefix='search_query: ',
                          cache_name='followup_hyp_query', batch_size=64)
T = np.stack([te[t] for t in held_texts]).astype(np.float32)
H = np.stack([he[h] for h in union_hyps]).astype(np.float32)
T /= np.linalg.norm(T, axis=1, keepdims=True)
H /= np.linalg.norm(H, axis=1, keepdims=True)
COS = T @ H.T
del te, he, T, H; ge.clear_mem()
COS.shape

## Cosine vs LLM Agreement

In [ ]:
llm = annotate_texts_with_concepts(texts=held_texts, concepts=union_hyps, model='Qwen/Qwen3-1.7B',
                                   cache_name='goemotions_heldout', n_workers=1,
                                   use_cache_only=True, uncached_value=0, max_words_per_example=64)

covered = float(np.mean([llm[h].std() > 0 for h in union_hyps]))
assert covered > 0, 'no cached annotations found; run notebook 02 first'

agree = {}
for j, h in enumerate(union_hyps):
    a = llm[h].astype(float)
    if a.std() > 0:
        agree[h] = float(pearsonr(COS[:, j], a)[0])
mean_agreement = float(np.mean(list(agree.values()))) if agree else None
ge.log_json('09_cosine_annotation', {'mean_pearson_cosine_vs_llm': mean_agreement,
                                     'per_hypothesis': agree})
mean_agreement

## Predictiveness Comparison

In [ ]:
def separation(a, y):
    p, n = a == 1, a == 0
    return float(y[p].mean() - y[n].mean()) if p.sum() and n.sum() else np.nan

llm_sep, cos_sep = [], []
for t, neurons in selected.items():
    y = held_targets[t].astype(float)
    for n in neurons:
        h = neuron2hyp.get(n)
        if not h:
            continue
        j = union_hyps.index(h)
        cos_bin = (COS[:, j] >= np.quantile(COS[:, j], 0.9)).astype(int)
        llm_sep.append(separation(llm[h], y))
        cos_sep.append(separation(cos_bin, y))
llm_sep, cos_sep = np.array(llm_sep), np.array(cos_sep)
mask = ~(np.isnan(llm_sep) | np.isnan(cos_sep))
rho = float(spearmanr(llm_sep[mask], cos_sep[mask]).correlation)
ge.log_json('09_cosine_annotation', {'spearman_predictiveness_llm_vs_cosine': rho})
rho

## Annotator Swap

In [ ]:
from google.colab import userdata

SWAP_MODELS = ['Qwen/Qwen3-0.6B', 'meta-llama/Llama-3.2-1B-Instruct']
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
except Exception:
    os.environ['HF_TOKEN'] = ''
if not os.environ.get('HF_TOKEN'):
    os.environ.pop('HF_TOKEN', None)          # an empty token breaks anonymous downloads
    SWAP_MODELS = [m for m in SWAP_MODELS if 'meta-llama' not in m]
    print('no HF_TOKEN; skipping gated models. SWAP_MODELS =', SWAP_MODELS)

ge.kill_stray_vllm()
print('free VRAM before swap (GB):', ge.gpu_free_gb())

In [ ]:
swap = {}
for model in SWAP_MODELS:
    proc = ge.serve_vllm(model, port=8001, max_model_len=2048, gpu_frac=0.60,
                         enforce_eager=True)
    try:
        kw = {'temperature': 0.0, 'max_output_tokens': 64}
        if 'Qwen3' in model:   # Llama has no thinking mode; disable Qwen's for a fair comparison
            kw['extra_body'] = {'chat_template_kwargs': {'enable_thinking': False}}
        ann = annotate_texts_with_concepts(texts=held_texts, concepts=union_hyps, model=model,
                                           cache_name='swap_' + model.split('/')[-1],
                                           n_workers=16, max_words_per_example=64, **kw)
        sig = {}
        for t, neurons in selected.items():
            hyps = sorted({neuron2hyp[n] for n in neurons if n in neuron2hyp})
            if len(hyps) < 2:
                continue
            m, _ = score_hypotheses({h: ann[h] for h in hyps},
                                    y_true=held_targets[t].astype(float), classification=True)
            sig[t] = m['Significant'][0]
        swap[model] = sig
    finally:
        ge.stop_vllm(proc)

ge.log_json('10_annotator_swap', {'significant_by_model': swap})
swap